# RescueVision - FloodNet transform repro (CPU)

In [ ]:
import subprocess, sys, os, json, time, traceback
from pathlib import Path

WORK = Path("/kaggle/working")
REPO = Path("/tmp/RescueVision")
if not REPO.exists():
    subprocess.run(["git", "clone", "-q", "https://github.com/AliOmarAbdelhady/RescueVision.git", str(REPO)], check=True)
sys.path.insert(0, str(REPO / "src"))

def mark(stage, ok=True, extra=""):
    """Progress ledger in /kaggle/working: survives kills where the log does not."""
    rec = {"t": time.strftime("%H:%M:%S"), "stage": stage, "ok": bool(ok), "extra": str(extra)[:1500]}
    with open(WORK / "_progress.txt", "a") as f:
        f.write(json.dumps(rec) + "\n")
    (WORK / "_status.json").write_text(json.dumps(rec, indent=1))
    print("[mark]", rec, flush=True)

mark("setup:start")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "albumentations"], check=False)

import torch
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# num_workers=0: DataLoader worker processes crash on the Kaggle GPU image
# (both floodnet v7 and v8 died at the first batch in a worker). Loading on the
# main process costs only a few percent of step time on /tmp-resident data.
NW = 0
try:
    torch.multiprocessing.set_sharing_strategy("file_system")
except Exception:
    pass
print("torch", torch.__version__, "| device:", DEVICE)
print(subprocess.run(["df", "-h", "/tmp", "/kaggle/working"], capture_output=True, text=True).stdout)
mark("setup:done", extra=str(DEVICE))

# per-epoch heartbeat: every trainer that logs an epoch also lands in the marks,
# so a silent kill still tells us how far training got and with what metrics
import rescuevision.training.logging_utils as _lu
_orig_log = _lu.CSVLogger.log
def _marked_log(self, metrics, epoch):
    try:
        mark(f"epoch{epoch}", extra=str({k: round(float(v), 4) for k, v in metrics.items()
                                         if isinstance(v, (int, float))})[:400])
    except Exception:
        pass
    return _orig_log(self, metrics, epoch)
_lu.CSVLogger.log = _marked_log
print("epoch heartbeat patch applied")

In [ ]:
import numpy as np, random, traceback
from PIL import Image
from pathlib import Path
import albumentations as A
print("albumentations", A.__version__)
mark("repro:start", extra=A.__version__)

ROOT = None
for _c in Path("/kaggle/input").rglob("class_mapping.csv"):
    ROOT = _c.parent
    break
assert ROOT is not None, "FloodNet dataset not found"
print("ROOT", ROOT)

# ---- exactly the tiling path, same split sampling as the trainer ----
TILE = 512
SEED = 42
VAL_FRAC = 0.15
TILES = Path("/tmp/floodnet_tiles")
for s in ["train", "val"]:
    (TILES / s).mkdir(parents=True, exist_ok=True)

img_dirs = [ROOT / "Train/Labeled/Flooded", ROOT / "Train/Labeled/Non-Flooded"]
samples = []
for d in img_dirs:
    for img_path in sorted((d / "image").glob("*.jpg")):
        mask_path = d / "mask" / (img_path.stem + "_lab.png")
        if not mask_path.exists():
            mask_path = d / "mask" / (img_path.stem + ".png")
        if mask_path.exists():
            samples.append((img_path, mask_path))
print(len(samples), "labeled images")

rng = random.Random(SEED)
val_set = set(rng.sample(range(len(samples)), max(1, int(len(samples) * VAL_FRAC))))

from rescuevision.constants import NUM_FLOODNET_CLASSES
n = 0
for i, (img_path, mask_path) in enumerate(samples):
    if n >= 20:      # only need enough for the first batches
        break
    split = "val" if i in val_set else "train"
    img = np.array(Image.open(img_path).convert("RGB"))
    sm = Image.open(mask_path)
    grayscale_mode = sm.mode in ("L", "P")
    ids = np.array(sm.convert("L"), dtype=np.uint8) if grayscale_mode else np.zeros(img.shape[:2], np.uint8)
    ids = np.clip(ids, 0, NUM_FLOODNET_CLASSES - 1)
    if i == 0:
        print("sample0: img", img.shape, img.dtype, "| mask mode", sm.mode, sm.size,
              "| ids", ids.shape, ids.dtype)
    H, W = img.shape[:2]
    for y in range(0, H - TILE + 1, TILE):
        for x in range(0, W - TILE + 1, TILE):
            tile_ids = ids[y:y+TILE, x:x+TILE]
            if len(np.unique(tile_ids)) < 2:
                continue
            stem = f"{img_path.stem}_{y//TILE}_{x//TILE}"
            np.save(TILES / split / f"{stem}_img.npy", img[y:y+TILE, x:x+TILE])
            np.save(TILES / split / f"{stem}_ids.npy", tile_ids)
            n += 1
print("saved", n, "tiles")

# ---- the EXACT dataset class ----
import torch
from torch.utils.data import Dataset, DataLoader
class FloodNetTileDataset(Dataset):
    def __init__(self, split_dir, transform=None):
        self.files = sorted(split_dir.glob("*_img.npy"))
        self.transform = transform
    def __len__(self):
        return len(self.files)
    def __getitem__(self, i):
        base = str(self.files[i])[: -len("_img.npy")]
        image = np.load(base + "_img.npy")
        mask = np.load(base + "_ids.npy")
        if self.transform:
            aug = self.transform(image=image, mask=mask)
            image, mask = aug["image"], aug["mask"]
        mask = mask.long() if torch.is_tensor(mask) else torch.from_numpy(mask).long()
        return {"image": image, "mask": mask, "tile_id": Path(base).name}

from rescuevision.data.transforms import get_segmentation_transforms
AUG = {"horizontal_flip": 0.5, "vertical_flip": 0.5, "rotate90": 0.5, "brightness_contrast": 0.3, "normalize": True}
tf = get_segmentation_transforms(AUG, "train", 512)
print("transform pipeline:", tf)

ds = FloodNetTileDataset(TILES / "train", tf)
print("train tiles:", len(ds), "first stems:", [Path(f).name for f in ds.files[:4]])

# ---- run the first FOUR tiles individually: full exception, no elision ----
for k in range(min(4, len(ds))):
    base = str(ds.files[k])[: -len("_img.npy")]
    image = np.load(base + "_img.npy")
    mask = np.load(base + "_ids.npy")
    print(f"tile {k}: image {image.shape} {image.dtype} | mask {mask.shape} {mask.dtype} | ndim {image.ndim}/{mask.ndim}")
    try:
        out = tf(image=image, mask=mask)
        print(f"tile {k}: TRANSFORM OK -> img {tuple(out['image'].shape)} {out['image'].dtype} mask {tuple(out['mask'].shape)}")
    except Exception as e:
        print(f"tile {k}: TRANSFORM FAIL -> {type(e).__name__}: {e}")
        mark(f"repro:fail-tile{k}", extra=f"{type(e).__name__}: {e}")
        try:
            tf2 = A.Compose(tf.transforms, is_check_shapes=False)
            out = tf2(image=image, mask=mask)
            print(f"tile {k}: FIX is_check_shapes=False OK -> {tuple(out['image'].shape)} {tuple(out['mask'].shape)}")
            mark(f"repro:fix1-ok-tile{k}")
        except Exception as e2:
            print(f"tile {k}: fix1 FAIL -> {type(e2).__name__}: {e2}")
            mark(f"repro:fix1-fail-tile{k}", extra=f"{type(e2).__name__}: {e2}")

# ---- batch through DataLoader like the probe ----
try:
    dl = DataLoader(ds, batch_size=4, num_workers=0)
    b = next(iter(dl))
    print("BATCH OK", b["image"].shape, b["image"].dtype, b["mask"].dtype)
    mark("repro:batch-ok")
except Exception as e:
    print("BATCH FAIL:", type(e).__name__, str(e)[:1000])
    mark("repro:batch-fail", extra=f"{type(e).__name__}: {str(e)[:1000]}")

# ---- oddity scan over ALL images (mode/size outliers) ----
odd = []
for grp in ["Flooded", "Non-Flooded"]:
    dd = ROOT / "Train/Labeled" / grp
    for p in sorted((dd / "image").glob("*.jpg")):
        im = Image.open(p)
        if im.mode != "RGB" or im.size[0] < TILE or im.size[1] < TILE:
            odd.append((grp, p.name, im.mode, im.size))
print("odd images:", odd[:10], "total", len(odd))
mark("repro:done", extra=f"odd={odd[:3]}")